<p align="center">
  <strong>Run this notebook:</strong>
</p>

<p align="center">
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/main/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://colab.research.google.com/assets/colab-badge.svg"
      alt="Open the main version in Google Colab"
    >
  </a>
  &nbsp;
  <a href="https://colab.research.google.com/github/MeteoSwiss/nwp-fdb-polytope-demo/blob/intro_earthkit_notebook/examples/earthkit/earthkit_onboarding_meteoswiss.ipynb">
    <img
      src="https://img.shields.io/badge/Open_dev_version-in%20Colab-F9AB00?logo=googlecolab&logoColor=white"
      alt="Open the development version in Google Colab"
    >
  </a>
</p>

<br>

<p align="center">
  <a href="https://earthkit.ecmwf.int/">
    <img
      src="https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-light.svg"
      alt="earthkit"
      width="520"
    >
  </a>
</p>

<h1 align="center">earthkit onboarding for MeteoSwiss NWP workflows</h1>

> **What is earthkit?**
>
> earthkit is an open-source Python ecosystem led by ECMWF. It provides a
> consistent workflow for accessing, inspecting, processing, analysing, and
> visualising weather and climate data—including the GRIB data commonly used
> in numerical weather prediction.

<p align="center">
  <a href="https://earthkit.ecmwf.int/">Website</a>
  &nbsp;·&nbsp;
  <a href="https://earthkit.readthedocs.io/en/latest/">Documentation</a>
  &nbsp;·&nbsp;
  <a href="https://github.com/ecmwf/earthkit">GitHub</a>
</p>

---

<p align="center">
  <img
    src="https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/intro_earthkit_notebook/examples/earthkit/earthkit_components.png"
    alt="Overview of earthkit components"
    width="720"
  >
</p>

## earthkit components

| Package | Purpose |
|---|---|
| [`earthkit-data`](https://earthkit-data.readthedocs.io/en/stable/) | A format-agnostic Python interface for geospatial data, with a focus on meteorology and climate science. |
| [`earthkit-plots`](https://earthkit-plots.readthedocs.io/en/stable/) | Produce publication-quality weather and climate charts and maps with only a few lines of code. |
| [`earthkit-meteo`](https://earthkit-meteo.readthedocs.io/en/stable/) | Perform common meteorological calculations using NumPy, Torch, CuPy, xarray, or field lists. |
| [`earthkit-geo`](https://earthkit-geo.readthedocs.io/en/stable/) | Work with geospatial shapes, coordinates, grids, and projections. |
| [`earthkit-transforms`](https://earthkit-transforms.readthedocs.io/en/stable/) | Apply transformations, aggregations, and statistical analyses across data cubes. |
| [`earthkit-hydro`](https://earthkit-hydro.readthedocs.io/en/stable/) | Work with river networks, flow accumulation, and other hydrological data. |

---

TODO table of content

## Why earthkit at MeteoSwiss?

MeteoSwiss uses gridded meteorological data across forecasting, research and machine-learning workflows. Many of these workflows repeat similar tasks, such as reading data, inspecting metadata, converting formats, regridding and plotting.

Using shared earthkit components can help to reduce duplicated MeteoSwiss-specific implementations and align with tools used across the meteorological community.

The release of earthkit 1.0 on 2 July 2026 marked its core interfaces as stable for wider research and operational use.

![earthkit-data-logo](https://github.com/ecmwf/logos/raw/refs/heads/main/logos/earthkit/earthkit-data-light.svg)

## The mental model to remember

The most useful earthkit mental model is:

```text
source → earthkit data object → FieldList / Xarray / NumPy / Pandas → analysis or plotting
```

## Environment setup

In [ ]:
!pip install earthkit==1.0.0 earthkit-plots

In [ ]:
import earthkit.data as ekd

print("earthkit-data:", ekd.__version__)

## Read a MeteoSwiss forecast

This example uses a small subset of an ICON-CH2-EPS forecast produced by MeteoSwiss:

- one forecast step;
- one variable: 2 m air temperature;
- the control member only.

The data is read directly from a URL. With `from_source()`, the source could also be a local file, FDB, Polytope or another supported input.

[See all supported sources](https://earthkit-data.readthedocs.io/en/latest/concepts/inputs/from_source.html#from_source)

In [ ]:
import earthkit.data as ekd

GRIB_URL = (
    "https://raw.githubusercontent.com/"
    "MeteoSwiss/nwp-fdb-polytope-demo/"
    "intro_earthkit_notebook/"
    "examples/earthkit/"
    "icon-ch2-eps-202607131200-100-t_2m-ctrl.grib2"
)

data = ekd.from_source("url", GRIB_URL)
data

The returned object provides some basic information but its primary goal is to convert the data into the required representation for further work. The actual data loading is deferred as much as possible, until the data is converted into a given type.

In [ ]:
print("Available conversions:", data.available_types)

### Fieldlists and fields
GRIB data can be converted into a [FieldList](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList), which represents each GRIB message as a field. In earthkit a [field](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field) is a horizontal slice of the atmosphere at a given time. In this sense the Field object is generic enough to represent other types of data than GRIB (e.g. NetCDF, GeoTIFF, dictionary data etc.)


In [ ]:
# fl is a fieldlist
fl = data.to_fieldlist()
print("Number of fields/messages:", len(fl))


[ls()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.ls) lists the fields in the fieldlist.

In [ ]:
fl.ls()

In [ ]:
# f is the first field in the fieldlist
field = fl[0]
print(field)

#### Field metadata

Each `Field` exposes format-independent metadata grouped into logical components.

| Component | Common keys |
|---|---|
| **Parameter** | `parameter.variable`, `parameter.units`, `parameter.chem_variable` |
| **Time** | `time.base_datetime`, `time.valid_datetime`, `time.step` |
| **Vertical** | `vertical.level`, `vertical.level_type` |
| **Geography** | `geography.latitudes`, `geography.longitudes`, `geography.shape` |
| **Ensemble** | `ensemble.member` |

Metadata values can be accessed with the [`get()`](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.get) method.


In [ ]:
print("parameter.variable:  ", field.get("parameter.variable"))
print("parameter.units:  ", field.get("parameter.units"))
print("time.base_datetime:  ", field.get("time.base_datetime"))
print("...")

More examples about discovering metadata: [2026-earthkit-training earthkit-data notebook](https://github.com/ecmwf-training/2026-earthkit-training/blob/main/content/earthkit-data/ekd-2-grib.ipynb)

#### Field overview

To get an overview about the components/keys of a field simply use the automatic display.

In [ ]:
field

Among the metadata, the **geography** section contains information about the underlying model grid, including a unique grid identifier (`uid`), the grid name, its extent (`area`), and the number of grid cells (`shape`).


In [ ]:
field.geography.grid_spec()

For more information about the ICON-CH2-EPS configuration, see the [Open Data documentation](https://opendatadocs.meteoswiss.ch/e-forecast-data/e2-e3-numerical-weather-forecasting-model).

#### Field geography

We can use [latlons()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/field/component/geography/index.html#earthkit.data.field.component.geography.GeographyBase.latlons) to retrieve the latitude and longitude arrays for all grid cells.

This is especially useful for ICON data because ICON uses an unstructured triangular grid rather than a regular latitude–longitude grid. The grid points are therefore not arranged in simple rows and columns, which makes their coordinates less obvious. `latlons()` provides the geographic position of each cell directly.

In [ ]:
lat, lon = field.geography.latlons()
print("latitudes:", lat)
print("longitudes:", lon)

#### Field values

Use [to_numpy](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.to_numpy) to access the field values as a numpy array. By default it uses the shape of the field.
The [values](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.values) property is also available. Please note this is always return a flattened array

In [ ]:
a = field.to_numpy()
a.shape, a

## Modifying fields

Fields can be modified by [set()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/field/index.html#earthkit.data.core.field.Field.set), which generates a new field with updated values/components.

In [ ]:
vals = field.values + 2.0
f1 = field.set({"vertical.level": 700, "values": vals})
f1.ls()

In [ ]:
# compare the metadata in the old and new fields
field.get("vertical.level"), f1.get("vertical.level")

In [ ]:
# compare the values in the old and new fields
field.values.max(), f1.values.max()

### Field selection

Use [sel()](https://earthkit-data.readthedocs.io/en/latest/autoapi/earthkit/data/core/fieldlist/index.html#earthkit.data.core.fieldlist.FieldList.sel) to select the fields matching the given metadata conditions.

In [ ]:
fl.sel({"parameter.variable": "2t"}).ls()

Examples about fields used in arithmetics: [2026-earthkit-training earthkit-data notebook](https://github.com/ecmwf-training/2026-earthkit-training/blob/main/content/earthkit-data/ekd-2-grib.ipynb)

## Converting to Xarray

We can convert a `FieldList` to an Xarray `Dataset` using earthkit-data's own
[Xarray engine](https://earthkit-data.readthedocs.io/en/latest/concepts/xarray/overview.html).
For most use cases, the default conversion is sufficient.

In [ ]:
ds = data.to_xarray()
ds


### Using Xarray profiles

The Xarray engine supports [**profiles**](https://earthkit-data.readthedocs.io/en/latest/concepts/xarray/overview.html#profiles), which control how the resulting
`Dataset` is organised. A profile can change, for example, how dimensions are
named, how variables are grouped, or how metadata is exposed.

The default profile is suitable for most applications, but other profiles may
produce a layout that is more convenient for specific workflows.

In [ ]:
ds_grib = data.to_xarray(profile="grib")
ds_mars = data.to_xarray(profile="mars")


Compare the dimensions, coordinates and variables of the two datasets to see
how the profile changes the Xarray representation.


In [ ]:
print("MARS profile")
display(ds_mars)

In [ ]:
print("GRIB profile")
display(ds_grib)

The two profiles produce the same data layout for this ICON field. The main difference is in the dataset-level metadata: the `mars` profile adds MARS-specific attributes such as `levtype`, `date`, `time`, and `number`, while the `grib` profile keeps a more minimal set of attributes. The profile therefore changes how metadata is represented, not the underlying data values. One can also define a custom profile.

### `earthkit` accessor

Xarrays created with earthkit-data have the `earthkit` accessor. It is an experimental feature and for each DataArray it stores earthkit specific metadata.

In [ ]:
ds["2t"].earthkit.grid_spec

## Plotting the ICON field

`earthkit-plots` can plot the earthkit-data `Field` directly. It uses the field metadata to identify the coordinates, variable and units automatically.

Because ICON uses an unstructured triangular grid, the values are not arranged in a regular latitude–longitude array. For a filled map, `earthkit-plots` interpolates the grid points onto the map projection for visualisation.


In [ ]:
import earthkit.plots as ekp

chart = ekp.Map(domain=[5.5, 10.8, 45.5, 48.2])
chart.plot(field, units="celsius")
chart.legend()
chart.coastlines()
chart.borders()
chart.gridlines()
chart.title("ICON-CH2-EPS {variable_name} – {time:%Y-%m-%d %H:%M UTC}")
chart.show()


## Further learning and examples

### Core documentation

- [earthkit](https://earthkit.readthedocs.io/en/latest/)
- [earthkit-data](https://earthkit-data.readthedocs.io/en/latest/)
- [earthkit-plots](https://earthkit-plots.readthedocs.io/en/latest/)
- [earthkit-geo](https://earthkit-geo.readthedocs.io/en/latest/)
- [earthkit-data 1.0 migration guide](https://earthkit-data.readthedocs.io/en/latest/release-notes/migration_1.0.0.html)

### Training and notebooks

- [ECMWF 2026 earthkit training](https://github.com/ecmwf-training/2026-earthkit-training/tree/main)
- [MeteoSwiss ICON-CH2 pollen forecast](https://github.com/MeteoSwiss/opendata-nwp-demos/blob/main/10_icon_ch2_pollen_forecast.ipynb)
- [MeteoSwiss Polytope polygon cut-out example](https://htmlpreview.github.io/?https://raw.githubusercontent.com/MeteoSwiss/nwp-fdb-polytope-demo/main/examples/snapshots/feature_polygon_country_cut-out.html)


TODOs:

- local eccodes definitions
- regridding
- plotting